# Replicating: *On the Effectiveness of LLMs in Writing Alloy Formulas*

**Hong, Jiang, Fu & Khurshid (2025)**

This notebook replicates the full experimental pipeline:
1. **Task 1 — English → Alloy**: Translate natural language to Alloy formula
2. **Task 2 — Alloy → Alloy**: Generate an equivalent alternative formula
3. **Task 3 — Sketch → Alloy**: Fill holes in an incomplete formula

Validation uses the Alloy Analyzer (SAT-backed equivalence checking via `iff`).

## 0. Setup & Configuration

In [2]:
import json, os, re, subprocess, tempfile, itertools, time
from pathlib import Path
from dataclasses import dataclass, field, asdict
from typing import Optional

import numpy as np
import pandas as pd
from openai import OpenAI
from dotenv import load_dotenv
from tqdm.notebook import tqdm

load_dotenv()
print("Libraries loaded.")

Libraries loaded.


In [11]:
# ── Configuration ──────────────────────────────────────────────────────────
# Adjust these before running the experiment.

ALLOY_PATH = "alloy"                      # Path to Alloy CLI executable (must be on PATH or absolute)
ALLOY_TIMEOUT = 30                         # Timeout in seconds for each Alloy check
ALLOY_SCOPE = 4                            # Scope bound for Alloy check commands

NUM_SOLUTIONS = 10                         # Number of solutions to generate per property per task
TEMPERATURE = 1.0                          # LLM sampling temperature

DATA_DIR = Path("data")                    # Directory containing .jsonl problem files
OUTPUT_DIR = Path("output")                # Directory for generated .als files and results
OUTPUT_DIR.mkdir(exist_ok=True)

# ── LLM Provider Configs ───────────────────────────────────────────────────
# Each entry: (display_name, base_url, api_key_env_var, model_id)
# Uses the OpenAI-compatible API format.  Add/remove models freely.

LLM_CONFIGS = [
    {
        "name": "gpt-4o",
        "base_url": None,                                 # default OpenAI
        "api_key_env": "OPENAI_API_KEY",
        "model": "gpt-4o",
    },
    # {
    #     "name": "deepseek-r1",
    #     "base_url": "https://api.deepseek.com/v1",
    #     "api_key_env": "DEEPSEEK_API_KEY",
    #     "model": "deepseek-reasoner",
    # },
    # {
    #     "name": "o4-mini",
    #     "base_url": None,
    #     "api_key_env": "OPENAI_API_KEY",
    #     "model": "o4-mini",
    # },
]

print(f"Models to evaluate: {[c['name'] for c in LLM_CONFIGS]}")
print(f"Solutions per property per task: {NUM_SOLUTIONS}")

Models to evaluate: ['gpt-4o']
Solutions per property per task: 10


## 1. Data Structures & Helpers

In [5]:
@dataclass
class Property:
    """One of the 11 subject specifications from the paper."""
    task_id: str
    domain: str                    # "graph" or "relation"
    prompt: str                    # natural language description
    signatures: str                # Alloy sig block
    predicate_name: str
    canonical_formula: str         # ground-truth formula (body only)
    sketch: str                    # formula with holes (_)
    sketch_hint: str               # hint about what to fill


@dataclass
class TrialResult:
    """Result of one LLM generation + Alloy validation."""
    task_id: str
    model: str
    task_type: str                 # "nl2alloy", "alloy2alloy", or "sketch2alloy"
    trial_idx: int
    raw_response: Optional[str] = None
    cleaned_formula: Optional[str] = None
    passed: bool = False
    error: Optional[str] = None    # None | "Syntax Error" | "Type Error" | "Counterexample" | ...


def load_properties(jsonl_path: Path) -> list[Property]:
    """Load properties from a JSONL file."""
    props = []
    with open(jsonl_path) as f:
        for line in f:
            props.append(Property(**json.loads(line)))
    return props


# Load all 11 properties
properties = (
    load_properties(DATA_DIR / "graph_properties.jsonl")
    + load_properties(DATA_DIR / "relation_properties.jsonl")
)
print(f"Loaded {len(properties)} properties:")
for p in properties:
    print(f"  [{p.domain:>8}] {p.task_id}: {p.prompt[:60]}")

Loaded 11 properties:
  [   graph] DAG: Directed acyclic graph
  [   graph] Cycle: Graph with directed cycle
  [   graph] Circular: The number of nodes is equal to the number of edges and the 
  [relation] Connex: For every pair of elements in S, either the first is related
  [relation] Reflexive: Every element in S is related to itself
  [relation] Symmetric: If element x in S is related to y, then y is also related to
  [relation] Transitive: If element x in S is related to y and y is related to z, the
  [relation] Antisymmetric: If element x in S is related to y and y is related to x, the
  [relation] Irreflexive: No element in S is related to itself
  [relation] Functional: Every element in S is related to at most one element (making
  [relation] Function: Every element in S is related to exactly one element (making


## 2. Alloy Validation Engine

In [6]:
def build_alloy_file(prop: Property, candidate_formula: str) -> str:
    """Build a complete .als file that checks equivalence of a candidate formula
    against the canonical ground truth using `iff`."""
    return f"""/* Auto-generated validation for {prop.task_id} */

{prop.signatures}

pred {prop.predicate_name} {{
\t{candidate_formula}
}}

check {prop.predicate_name} {{
    {prop.predicate_name} iff ({prop.canonical_formula})
}} for {ALLOY_SCOPE}
"""


def check_alloy(als_content: str, debug_path: Optional[Path] = None) -> tuple[bool, Optional[str]]:
    """Run the Alloy analyzer on an .als file and return (passed, error_msg).
    
    Returns:
        (True, None)  if UNSAT (no counterexample — formulas are equivalent)
        (False, msg)  otherwise
    """
    with tempfile.NamedTemporaryFile(suffix=".als", mode="w", delete=False) as f:
        f.write(als_content)
        tmp_path = f.name

    # Also save a permanent copy for debugging if requested
    if debug_path:
        debug_path.parent.mkdir(parents=True, exist_ok=True)
        debug_path.write_text(als_content)

    try:
        result = subprocess.run(
            [ALLOY_PATH, "exec", "-o", "/tmp", "-f", tmp_path],
            capture_output=True, text=True, timeout=ALLOY_TIMEOUT,
        )
        stderr = result.stderr
        if "UNSAT" in stderr:
            return True, None
        if "Syntax error" in stderr:
            return False, "Syntax Error"
        if "Type error" in stderr:
            return False, "Type Error"
        if "SAT" in stderr:
            return False, "Counterexample"
        return False, f"Unknown: {stderr[:200]}"
    except subprocess.TimeoutExpired:
        return False, "Timeout"
    except FileNotFoundError:
        return False, f"Alloy CLI not found at '{ALLOY_PATH}'"
    except Exception as e:
        return False, str(e)
    finally:
        try:
            os.unlink(tmp_path)
        except OSError:
            pass


# Quick smoke test: validate canonical formula against itself
p = properties[0]
als = build_alloy_file(p, p.canonical_formula)
ok, err = check_alloy(als)
print(f"Smoke test ({p.task_id} canonical vs. canonical): passed={ok}, error={err}")
if not ok:
    print("⚠  Alloy CLI may not be configured. Check ALLOY_PATH above.")
    print(f"   Generated file:\n{als}")

Smoke test (DAG canonical vs. canonical): passed=False, error=Unknown: [main] ERROR alloy - excuting sub command CLI:exec error java.nio.file.NoSuchFileException: /tmp/tmpb_n_m5t2.als
java.nio.file.NoSuchFileException: /tmp/tmpb_n_m5t2.als
	at java.base/sun.nio.fs.UnixEx
⚠  Alloy CLI may not be configured. Check ALLOY_PATH above.
   Generated file:
/* Auto-generated validation for DAG */

sig Node {
	link : set Node
}

pred DAG {
	all n: Node | n !in n.^ link
}

check DAG {
    DAG iff (all n: Node | n !in n.^ link)
} for 4



## 3. LLM Client (OpenAI-Compatible)

In [7]:
SYSTEM_PROMPT = (
    "You are an expert in formal methods and the Alloy specification language. "
    "When asked, output ONLY the Alloy formula (the predicate body), nothing else. "
    "Do not include the pred declaration, curly braces, or any explanation."
)


def make_client(config: dict) -> OpenAI:
    """Create an OpenAI-compatible client from a config dict."""
    kwargs = {"api_key": os.getenv(config["api_key_env"], "")}
    if config.get("base_url"):
        kwargs["base_url"] = config["base_url"]
    return OpenAI(**kwargs)


def query_llm(client: OpenAI, model: str, user_prompt: str) -> Optional[str]:
    """Send a single prompt to the LLM and return the raw text response."""
    try:
        resp = client.chat.completions.create(
            model=model,
            messages=[
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": user_prompt},
            ],
            temperature=TEMPERATURE,
            max_tokens=512,
        )
        return resp.choices[0].message.content
    except Exception as e:
        print(f"  LLM error: {e}")
        return None


def clean_formula(raw: Optional[str]) -> Optional[str]:
    """Extract and clean the Alloy formula from an LLM response."""
    if not raw:
        return None
    text = raw.strip()
    # Extract from markdown code block if present
    m = re.search(r"```(?:alloy)?\s*(.*?)```", text, re.DOTALL)
    if m:
        text = m.group(1).strip()
    # Remove pred wrapper if LLM included it
    m2 = re.search(r"pred\s+\w+\s*\{(.*)\}", text, re.DOTALL)
    if m2:
        text = m2.group(1).strip()
    # Remove stray trailing brace
    text = text.rstrip("}").strip()
    # Remove triple-quotes
    text = text.replace('"""', "").strip()
    return text if text else None


print("LLM client helpers defined.")

LLM client helpers defined.


## 4. Prompt Templates (Three Tasks)

In [8]:
def prompt_nl2alloy(prop: Property) -> str:
    """Task 1: Natural Language → Alloy."""
    return (
        f"Implement the following Alloy predicate {prop.predicate_name} "
        f"as defined in the comments:\n\n"
        f"{prop.signatures}\n"
        f"pred {prop.predicate_name} {{\n"
        f"  -- {prop.prompt}\n"
        f"}}\n\n"
        f"Output only the formula in the predicate body."
    )


def prompt_alloy2alloy(prop: Property) -> str:
    """Task 2: Alloy → Alloy (generate equivalent alternative)."""
    return (
        f"Given the following Alloy predicate:\n\n"
        f"{prop.signatures}\n"
        f"pred {prop.predicate_name} {{\n"
        f"  {prop.canonical_formula}\n"
        f"}}\n\n"
        f"Create an alternative but logically equivalent Alloy formula for this predicate.\n"
        f"Output only the formula in the predicate body."
    )


def prompt_sketch2alloy(prop: Property) -> str:
    """Task 3: Sketch → Alloy (fill in holes marked with _)."""
    return (
        f"Complete the following Alloy sketch to satisfy the property described in the comments:\n\n"
        f"{prop.signatures}\n"
        f"pred {prop.predicate_name} {{\n"
        f"  -- {prop.prompt}\n"
        f"  {prop.sketch}\n"
        f"}}\n\n"
        f"Hint: {prop.sketch_hint}\n"
        f"Populate the holes (marked with _) in the sketch. Output only the completed formula in the predicate body."
    )


TASK_CONFIGS = {
    "nl2alloy":     {"prompt_fn": prompt_nl2alloy,     "label": "Task 1: NL → Alloy"},
    "alloy2alloy":  {"prompt_fn": prompt_alloy2alloy,  "label": "Task 2: Alloy → Alloy"},
    "sketch2alloy": {"prompt_fn": prompt_sketch2alloy, "label": "Task 3: Sketch → Alloy"},
}

# Preview one prompt per task
for task_name, cfg in TASK_CONFIGS.items():
    print(f"\n{'='*60}\n{cfg['label']}\n{'='*60}")
    print(cfg["prompt_fn"](properties[0]))


Task 1: NL → Alloy
Implement the following Alloy predicate DAG as defined in the comments:

sig Node {
	link : set Node
}
pred DAG {
  -- Directed acyclic graph
}

Output only the formula in the predicate body.

Task 2: Alloy → Alloy
Given the following Alloy predicate:

sig Node {
	link : set Node
}
pred DAG {
  all n: Node | n !in n.^ link
}

Create an alternative but logically equivalent Alloy formula for this predicate.
Output only the formula in the predicate body.

Task 3: Sketch → Alloy
Complete the following Alloy sketch to satisfy the property described in the comments:

sig Node {
	link : set Node
}
pred DAG {
  -- Directed acyclic graph
  all n: Node | n _ n.^ link
}

Hint: Replace _ with the correct membership operator
Populate the holes (marked with _) in the sketch. Output only the completed formula in the predicate body.


## 5. Run Experiment

In [9]:
def run_experiment(
    properties: list[Property],
    llm_configs: list[dict],
    task_names: list[str] | None = None,
    num_solutions: int = NUM_SOLUTIONS,
    save_als: bool = True,
) -> list[TrialResult]:
    """
    Run the full experiment: for each model × task × property × trial,
    query the LLM, validate with Alloy, collect results.
    """
    if task_names is None:
        task_names = list(TASK_CONFIGS.keys())

    all_results: list[TrialResult] = []
    total = len(llm_configs) * len(task_names) * len(properties) * num_solutions
    pbar = tqdm(total=total, desc="Running experiment")

    for cfg in llm_configs:
        model_name = cfg["name"]
        client = make_client(cfg)
        model_id = cfg["model"]

        for task_name in task_names:
            prompt_fn = TASK_CONFIGS[task_name]["prompt_fn"]
            task_label = TASK_CONFIGS[task_name]["label"]

            for prop in properties:
                user_prompt = prompt_fn(prop)

                for trial_idx in range(num_solutions):
                    pbar.set_postfix_str(f"{model_name}/{task_name}/{prop.task_id}/#{trial_idx}")

                    # Query LLM
                    raw = query_llm(client, model_id, user_prompt)
                    formula = clean_formula(raw)

                    # Validate with Alloy
                    passed, error = False, "No formula generated"
                    if formula:
                        als_content = build_alloy_file(prop, formula)
                        debug_path = None
                        if save_als:
                            debug_path = (
                                OUTPUT_DIR / model_name / task_name
                                / f"{prop.task_id}_sol{trial_idx}.als"
                            )
                        passed, error = check_alloy(als_content, debug_path)

                    result = TrialResult(
                        task_id=prop.task_id,
                        model=model_name,
                        task_type=task_name,
                        trial_idx=trial_idx,
                        raw_response=raw,
                        cleaned_formula=formula,
                        passed=passed,
                        error=error,
                    )
                    all_results.append(result)
                    pbar.update(1)

    pbar.close()
    return all_results

In [12]:
# ── Run the full experiment ────────────────────────────────────────────────
# Set task_names=["nl2alloy"] to run only Task 1, etc.

results = run_experiment(
    properties=properties,
    llm_configs=LLM_CONFIGS,
    task_names=None,           # None = all three tasks
    num_solutions=NUM_SOLUTIONS,
)

print(f"\nTotal trials: {len(results)}")
print(f"Passed: {sum(r.passed for r in results)}")

Running experiment:   0%|          | 0/330 [00:00<?, ?it/s]

  LLM error: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}}


KeyboardInterrupt: 

In [ ]:
# ── Save raw results to JSON ──────────────────────────────────────────────

results_path = OUTPUT_DIR / "all_results.json"
with open(results_path, "w") as f:
    json.dump([asdict(r) for r in results], f, indent=2)
print(f"Saved {len(results)} results to {results_path}")

## 6. Analysis & Reporting

In [ ]:
# ── Load results into a DataFrame ─────────────────────────────────────────

df = pd.DataFrame([asdict(r) for r in results])
df.head()

In [ ]:
# ── Pass@k estimator (unbiased, as in the paper) ──────────────────────────

def pass_at_k(n: int, c: int, k: int) -> float:
    """Unbiased estimator: 1 - C(n-c, k) / C(n, k)."""
    if n - c < k:
        return 1.0
    return 1.0 - np.prod(1.0 - k / np.arange(n - c + 1, n + 1))


def compute_pass_at_k_table(df: pd.DataFrame, k_values: list[int] = [1, 5, 10]) -> pd.DataFrame:
    """Compute pass@k for each (model, task_type, task_id) group."""
    rows = []
    for (model, task_type, task_id), grp in df.groupby(["model", "task_type", "task_id"]):
        n = len(grp)
        c = int(grp["passed"].sum())
        row = {"model": model, "task_type": task_type, "task_id": task_id, "n": n, "c": c}
        for k in k_values:
            if k <= n:
                row[f"pass@{k}"] = pass_at_k(n, c, k)
        rows.append(row)
    return pd.DataFrame(rows)

In [ ]:
# ── Success Rate Table (per model × task) ─────────────────────────────────

summary = (
    df.groupby(["model", "task_type"])
    .agg(total=("passed", "count"), passed=("passed", "sum"))
    .assign(rate=lambda x: (x["passed"] / x["total"] * 100).round(1))
)
print("\n── Overall Success Rates ──")
display(summary)

In [ ]:
# ── Detailed per-property success rate ────────────────────────────────────

detail = (
    df.groupby(["model", "task_type", "task_id"])
    .agg(total=("passed", "count"), passed=("passed", "sum"))
    .assign(rate=lambda x: (x["passed"] / x["total"] * 100).round(1))
)
print("\n── Per-Property Success Rates ──")
display(detail)

In [ ]:
# ── Pass@k Table ──────────────────────────────────────────────────────────

k_values = [k for k in [1, 5, 10] if k <= NUM_SOLUTIONS]
pak = compute_pass_at_k_table(df, k_values)

# Average pass@k per model × task
pak_summary = (
    pak.groupby(["model", "task_type"])[[f"pass@{k}" for k in k_values]]
    .mean()
    .round(3)
)
print("\n── Average Pass@k ──")
display(pak_summary)

In [ ]:
# ── Error Classification ──────────────────────────────────────────────────

error_counts = (
    df[~df["passed"]]
    .groupby(["model", "task_type", "error"])
    .size()
    .reset_index(name="count")
    .sort_values(["model", "task_type", "count"], ascending=[True, True, False])
)
print("\n── Error Breakdown (failed trials only) ──")
display(error_counts)

In [ ]:
# ── Diversity: unique correct formulas per property ────────────────────────

diversity = (
    df[df["passed"]]
    .groupby(["model", "task_type", "task_id"])["cleaned_formula"]
    .nunique()
    .reset_index(name="unique_correct")
)
print("\n── Diversity: Unique Correct Formulas ──")
display(diversity)

## 7. Visualization

In [ ]:
import matplotlib.pyplot as plt

# ── Bar chart: success rate by model × task ───────────────────────────────

fig, axes = plt.subplots(1, len(TASK_CONFIGS), figsize=(5 * len(TASK_CONFIGS), 5), sharey=True)
if len(TASK_CONFIGS) == 1:
    axes = [axes]

for ax, (task_name, cfg) in zip(axes, TASK_CONFIGS.items()):
    task_df = df[df["task_type"] == task_name]
    if task_df.empty:
        continue
    pivot = (
        task_df.groupby(["task_id", "model"])["passed"]
        .mean()
        .unstack("model")
        .reindex([p.task_id for p in properties])
        .dropna(how="all")
        * 100
    )
    pivot.plot.bar(ax=ax, rot=45)
    ax.set_title(cfg["label"])
    ax.set_ylabel("Success Rate (%)")
    ax.set_ylim(0, 105)
    ax.legend(title="Model")

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "success_rates.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved figure to {OUTPUT_DIR / 'success_rates.png'}")

In [ ]:
# ── Heatmap: per-property pass@1 ──────────────────────────────────────────

fig, axes = plt.subplots(1, len(TASK_CONFIGS), figsize=(5 * len(TASK_CONFIGS), 6), sharey=True)
if len(TASK_CONFIGS) == 1:
    axes = [axes]

for ax, (task_name, cfg) in zip(axes, TASK_CONFIGS.items()):
    task_pak = pak[pak["task_type"] == task_name]
    if task_pak.empty or "pass@1" not in task_pak.columns:
        continue
    heatmap_data = task_pak.pivot(index="task_id", columns="model", values="pass@1")
    heatmap_data = heatmap_data.reindex([p.task_id for p in properties]).dropna(how="all")

    im = ax.imshow(heatmap_data.values, cmap="RdYlGn", vmin=0, vmax=1, aspect="auto")
    ax.set_xticks(range(len(heatmap_data.columns)))
    ax.set_xticklabels(heatmap_data.columns, rotation=45, ha="right")
    ax.set_yticks(range(len(heatmap_data.index)))
    ax.set_yticklabels(heatmap_data.index)
    ax.set_title(cfg["label"])

    # Annotate cells
    for i in range(len(heatmap_data.index)):
        for j in range(len(heatmap_data.columns)):
            val = heatmap_data.values[i, j]
            if not np.isnan(val):
                ax.text(j, i, f"{val:.0%}", ha="center", va="center", fontsize=9)

fig.colorbar(im, ax=axes, shrink=0.6, label="pass@1")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "pass_at_1_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()

## 8. Export Final Report

In [ ]:
# ── Export summary tables to CSV ──────────────────────────────────────────

summary.to_csv(OUTPUT_DIR / "summary_by_model_task.csv")
detail.to_csv(OUTPUT_DIR / "detail_by_property.csv")
pak_summary.to_csv(OUTPUT_DIR / "pass_at_k_summary.csv")
error_counts.to_csv(OUTPUT_DIR / "error_breakdown.csv", index=False)
diversity.to_csv(OUTPUT_DIR / "diversity.csv", index=False)

print("Exported CSV files to output/:")
for f in sorted(OUTPUT_DIR.glob("*.csv")):
    print(f"  {f.name}")

In [ ]:
# ── Final summary printout ────────────────────────────────────────────────

print("\n" + "="*60)
print(" EXPERIMENT COMPLETE")
print("="*60)
print(f" Properties:     {len(properties)}")
print(f" Models:         {[c['name'] for c in LLM_CONFIGS]}")
print(f" Tasks:          {list(TASK_CONFIGS.keys())}")
print(f" Trials/prop:    {NUM_SOLUTIONS}")
print(f" Total trials:   {len(results)}")
print(f" Total passed:   {sum(r.passed for r in results)}")
print(f" Overall rate:   {sum(r.passed for r in results)/len(results)*100:.1f}%")
print("="*60)